In [0]:
dbutils.widgets.text("init_load_flag","1")
init_load_flag=int(dbutils.widgets.get("init_load_flag"))

####Data reading from source

In [0]:
df=spark.sql("select * from databricks_catalog.silver.customers_silver")

In [0]:
df.display()

###Removing duplicatess

In [0]:
df=df.drop_duplicates(subset=['customer_id'])



In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
df_new=spark.sql("select * from databricks_catalog.silver.orders_silver")

###Divinding new vs old records

###Surrogate Key - All Values

In [0]:
if init_load_flag==0:
  df_old=spark.sql('''select DimCustomerKey, customer_id, create_date,update_date from databricks_catalog.gold.DimCustomers''')
  
else:
  df_old=spark.sql('''select 0 DimCustomerKey, 0 customer_id, 0 create_date,0 update_date from databricks_catalog.silver.customers_silver where 1==0''')

In [0]:
df_old.display()

In [0]:
df_old=df_old.withColumnRenamed("DimCustomerKey","old_DimCustomerKey")\
.withColumnRenamed("customer_id","old_customer_id")\
.withColumnRenamed("create_date","old_create_date")\
.withColumnRenamed("update_date","old_update_date")

#####Applying join with old records


In [0]:
df_join=df.join(df_old,df.customer_id==df_old.old_customer_id,"left")


In [0]:
df_join.display()

###old vs new

In [0]:
df_new=df_join.filter(df_join["old_DimCustomerKey"].isNull())


In [0]:
df_old=df_join.filter(df_join["old_DimCustomerKey"].isNotNull())


####preparing df_old

In [0]:
# Dropping all the columns which are not required
df_old=df_old.drop("old_customer_id","old_update_date")
# Adding update date column
df_old=df_old.withColumnRenamed("old_DimCustomerKey","DimCustomerKey")
df_old=df_old.withColumnRenamed("old_create_date","create_date")
df_old=df_old.withColumn("create_date",to_timestamp(col("create_date")))
# recreating update date column
df_old=df_old.withColumn("update_date",current_timestamp())





In [0]:
df_old.display()

###preparing df_new

In [0]:
# Dropping all the columns which are not required
df_new=df_new.drop("old_DimCustomerKey","old_customer_id","old_create_date","old_update_date")

# recreating update date and current_datecolumn

df_new=df_new.withColumn("update_date",current_timestamp())
df_new=df_new.withColumn("create_date",current_timestamp())







In [0]:
df_new.display()


###Surrogate Key 

In [0]:
df_new=df_new.withColumn("DimCustomerKey",monotonically_increasing_id()+lit(1))
df_new.display()

###Adding max surrogate key

In [0]:
if init_load_flag==1:
    max_surrogate_key=0
else:
    df_max_surrogate=spark.sql("select max(DimCustomerKey) as max_surrogate_key from databricks_catalog.gold.DimCustomers")
    #convertomg df_max_surrogate to max surrogate key variable
    max_surrogate_key=df_max_surrogate.collect()[0]['max_surrogate_key']
    

In [0]:
df_new=df_new.withColumn("DimCustomerKey",lit(max_surrogate_key)+col("DimCustomerKey"))


union of df_old and df_new

In [0]:
df_final=df_new.unionByName(df_old)


##SCD Type1

In [0]:
from delta.tables import DeltaTable
if spark.catalog.tableExists("datrabricks_catalog.silver.DimCustomers"):
    dlt_obj=DeltaTable.forPath(spark,"abfss://gold@datalakedatabricksram.dfs.core.windows.net/DimCustomers")
    dlt_obj.alis("trg").merge(df_final.alias("src"),"trg.DimCustomerKey=src.DimCustomerKey")\
        .whenMatchedUpdateAll().whenNotMatchedInsertAll()\
            .execute()
    
else:
    df_final.write.mode("overwrite")\
        .format("delta")\
        .option("path","abfss://gold@datalakedatabricksram.dfs.core.windows.net/DimCustomers")\
            .saveAsTable("databricks_catalog.gold.DimCustomers")
    

In [0]:
%sql
select * from databricks_catalog.gold.DimCustomers